# CSCIE89 HW05 – Problem 1: Image transformations of a tiger

We download a moderate-resolution photo of a tiger and use `torchvision.transforms.Compose()` to demonstrate five transformations:

1. Flip around the **horizontal** axis (upside-down).
2. Flip around the **vertical** axis (left-right mirror).
3. **10% stretch** along one diagonal.
4. **10° rotation to the left** (counter-clockwise).
5. **15° rotation to the right** (clockwise).

The code follows the structure of the flower-augmentation notebook from class.

## 1. Install packages and import tools

If the imports fail, uncomment the installation line, run it once, then restart the kernel.

In [ ]:
# %pip install torch torchvision matplotlib pillow

from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torchvision import transforms
from torchvision.datasets.utils import download_url

DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)
print('PyTorch:', torch.__version__)

## 2. Download a tiger image

We download a 1024-pixel-wide tiger photo from Wikimedia Commons (moderate resolution). The file is downloaded only once.

In [ ]:
tiger_path = DATA_DIR / 'tiger.jpg'
if not tiger_path.exists():
    download_url(
        'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Walking_tiger_female.jpg/1024px-Walking_tiger_female.jpg',
        root=str(DATA_DIR), filename='tiger.jpg',
    )
with Image.open(tiger_path) as picture:
    photo = picture.convert('RGB')  # Convert to three-channel RGB.
WIDTH, HEIGHT = photo.size
print('Image size (width x height):', WIDTH, 'x', HEIGHT)
plt.imshow(photo)
plt.title('Original tiger')
plt.axis('off')
plt.show()

## 3. Define the five transformations with `transforms.Compose()`

- **Flip around the horizontal axis:** `RandomVerticalFlip(p=1.0)` always turns the image upside-down.
- **Flip around the vertical axis:** `RandomHorizontalFlip(p=1.0)` always mirrors the image left-to-right.
- **10% stretch along one diagonal:** rotate the image 45° so the diagonal lies along the x-axis, stretch the width by 10%, rotate back by −45°, and crop the center to the original size.
- **10° rotation to the left:** in torchvision a positive angle is counter-clockwise, so we use `RandomRotation((10, 10))`.
- **15° rotation to the right:** a negative angle is clockwise, so we use `RandomRotation((-15, -15))`.

Using a fixed range such as `(10, 10)` makes the "random" rotation always use exactly that angle. Corners uncovered by a rotation are padded black. Every pipeline ends with `ToTensor()`, as in the class notebook.

In [ ]:
# Original image with no changes, for comparison.
image_transform = transforms.Compose([
    transforms.ToTensor(),
])

flip_horizontal_axis = transforms.Compose([
    transforms.RandomVerticalFlip(p=1.0),    # Upside-down: flip around the horizontal axis.
    transforms.ToTensor(),
])

flip_vertical_axis = transforms.Compose([
    transforms.RandomHorizontalFlip(p=1.0),  # Mirror: flip around the vertical axis.
    transforms.ToTensor(),
])

diagonal_stretch = transforms.Compose([
    transforms.RandomRotation((45, 45), expand=True),    # Put the diagonal along the x-axis.
    transforms.Lambda(lambda img: img.resize((int(img.width * 1.1), img.height))),  # Stretch 10%.
    transforms.RandomRotation((-45, -45), expand=True),  # Rotate back.
    transforms.CenterCrop((HEIGHT, WIDTH)),              # Back to the original size.
    transforms.ToTensor(),
])

rotate_left_10 = transforms.Compose([
    transforms.RandomRotation((10, 10)),     # Positive angle = counter-clockwise (left).
    transforms.ToTensor(),
])

rotate_right_15 = transforms.Compose([
    transforms.RandomRotation((-15, -15)),   # Negative angle = clockwise (right).
    transforms.ToTensor(),
])

## 4. Show the original and the five transformed images

In [ ]:
views = [
    ('Original', image_transform),
    ('Flip around horizontal axis', flip_horizontal_axis),
    ('Flip around vertical axis', flip_vertical_axis),
    ('10% stretch along diagonal', diagonal_stretch),
    ('10° rotation to the left', rotate_left_10),
    ('15° rotation to the right', rotate_right_15),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for axis, (title, transform) in zip(axes.flat, views):
    picture = transform(photo)
    axis.imshow(picture.permute(1, 2, 0))  # Plotting uses height, width, channels.
    axis.set_title(title)
    axis.axis('off')
fig.suptitle('Same tiger, five transformations')
plt.tight_layout()
plt.show()

## Summary

- We downloaded a moderate-resolution tiger image.
- We used `transforms.Compose()` to build five transformation pipelines: two flips, a 10% diagonal stretch, a 10° left rotation, and a 15° right rotation.
- Each pipeline ends with `ToTensor()`, and the results are plotted next to the original image.